---

![alt text](https://storage.googleapis.com/kaggle-datasets-images/10074547/15724359/c9f62588cfe1713b0398d714166a0de1/dataset-cover.jpg?t=2026-04-14-05-41-39 "Title")

# Ember Global Electricity Data: Monthly Long Format

---

#### About Dataset


This dataset provides a comprehensive, long-format time-series of global monthly electricity statistics. It covers electricity generation, power sector emissions, and generating capacity across more than 80 countries and regions.

The data is meticulously curated by Ember, an independent energy think tank, and offers a granular look at the global transition toward renewable energy and the reduction of fossil fuel reliance.

---

#### Job List

- Load Data
- Ekstrak format Date
- Cek data kosong dan data duplikat
- Cek tipe data dan perbaikan tipe data jika memungkinkan
- Menghindari double counting
- Memisahkan antara region dan country
- Buat label apakah provinsional atau bukan
- Pengecekan ketersediaan data
- Cek outlier
- Ringkasan akhir beserta simpan data baru
- Langsung saja reg dan Queen

---

### Setup Liblary 

In [ ]:
import pandas as pd
import numpy as np
import warnings

# Menonaktifkan warning agar output lebih bersih
warnings.filterwarnings('ignore')

### Load Data

In [ ]:
# Path menuju file dataset mentah
lokasi_file = 'dataset/monthly_full_release_long_format.csv'

# Membaca CSV menjadi DataFrame
data_mentah = pd.read_csv(lokasi_file)

print(f"Dataset berhasil dimuat dari: {lokasi_file}")
print(f"Jumlah baris   : {data_mentah.shape[0]:,}")
print(f"Jumlah kolom   : {data_mentah.shape[1]}")
print(f"Kolom          : {data_mentah.columns.tolist()}")
print(f"\nPreview 5 baris pertama:")
print(data_mentah.head().to_string())

### Ekstrak Format Date

Penjelasan:
- Kolom 'Date' dikonversi dari string (teks) ke tipe datetime.
- Diekstrak komponen: Year (tahun), Month (bulan), Quarter (kuartal), YearMonth (tahun-bulan).
- Ini memudahkan analisis time-series dan pengelompokan per periode.

In [ ]:
# Konversi kolom 'Date' dari string ke tipe datetime
data_mentah['Date'] = pd.to_datetime(data_mentah['Date'])

# Ekstrak komponen waktu dari kolom Date
data_mentah['Year'] = data_mentah['Date'].dt.year         # Tahun (misal: 2024)
data_mentah['Month'] = data_mentah['Date'].dt.month       # Bulan (1-12)
data_mentah['Quarter'] = data_mentah['Date'].dt.quarter   # Kuartal (1-4, tiap 3 bulan)
data_mentah['YearMonth'] = data_mentah['Date'].dt.strftime('%Y-%m')  # Format "2024-01"

print(f"Tipe kolom Date : {data_mentah['Date'].dtype}")
print(f"Rentang data    : {data_mentah['Date'].min().date()} s/d {data_mentah['Date'].max().date()}")
print(f"Kolom baru      : Year, Month, Quarter, YearMonth")
print(f"\nSample hasil ekstraksi:")
print(data_mentah[['Date', 'Year', 'Month', 'Quarter', 'YearMonth']].head(10).to_string())

### Cek data Kosong

Penjelasan:
- Mengecek jumlah missing value (nilai kosong) per kolom.
- Mengecek apakah ada baris duplikat (baris yang isinya 100% sama).
- Kolom seperti 'ISO 3 code', 'Continent', dll. kosong untuk tipe Region
  karena Region bukan negara tunggal.
- Kolom 'YoY' (Year-over-Year) kosong untuk tahun pertama karena belum
  ada data tahun sebelumnya sebagai pembanding.

In [ ]:
# Hitung jumlah nilai kosong per kolom
jumlah_kosong = data_mentah.isnull().sum()

# Hitung persentase nilai kosong per kolom
persentase_kosong = (data_mentah.isnull().sum() / len(data_mentah) * 100).round(2)

# Gabungkan ke tabel ringkasan
ringkasan_kosong = pd.DataFrame({
    'Jumlah Null': jumlah_kosong,
    'Persentase (%)': persentase_kosong
})

print("Missing values per kolom:")
# Tampilkan hanya kolom yang memiliki nilai kosong
print(ringkasan_kosong[ringkasan_kosong['Jumlah Null'] > 0].to_string())

### Cek data duplikat

In [ ]:
# Hitung jumlah baris yang 100% identik dengan baris lain
jumlah_duplikat = data_mentah.duplicated().sum()
print(f"\nJumlah baris duplikat: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    # Hapus baris duplikat, simpan hanya kemunculan pertama
    data_mentah = data_mentah.drop_duplicates()
    print(f"Duplikat telah dihapus. Sisa baris: {len(data_mentah):,}")
else:
    print("Tidak ada duplikat ditemukan. Data bersih.")

### Cek tipe data dan perbaikan

Penjelasan:
- Memastikan setiap kolom memiliki tipe data yang tepat.
- Kolom flag keanggotaan (EU, OECD, G20, G7, ASEAN) awalnya bertipe float
  karena ada NaN. Dikonversi ke boolean (True/False) agar jelas.
- Kolom kategorikal (Area, Category, dll.) dikonversi ke tipe 'category'
  untuk menghemat memori.

Perbaiki kolom flag keanggotaan

In [ ]:
# Kolom ini berisi 0.0 / 1.0 / NaN, diubah jadi True/False
# NaN (untuk Region) diisi 0 dulu, baru dikonversi
kolom_flag = ['EU', 'OECD', 'G20', 'G7', 'ASEAN']
for nama_kolom in kolom_flag:
    data_mentah[nama_kolom] = (
        data_mentah[nama_kolom]
        .fillna(0)       # Isi NaN dengan 0
        .astype(int)     # float -> int (0 atau 1)
        .astype(bool)    # int -> bool (False atau True)
    )

Perbaiki kolom kategorikal

In [ ]:
# Tipe 'category' lebih hemat memori daripada 'object' (string biasa)
kolom_kategorikal = [
    'Area', 'Area type', 'Continent', 'Ember region',
    'Category', 'Subcategory', 'Variable', 'Unit'
]
for nama_kolom in kolom_kategorikal:
    data_mentah[nama_kolom] = data_mentah[nama_kolom].astype('category')

print("\nTipe data SETELAH perbaikan:")
print(data_mentah.dtypes.to_string())

# Cek penggunaan memori setelah optimasi
memori_mb = data_mentah.memory_usage(deep=True).sum() / 1e6
print(f"\nMemori digunakan: {memori_mb:.1f} MB")

### Hindari double counting

Penjelasan:
- Dataset punya Subcategory 'Aggregate fuel' dan 'Fuel'.
  'Aggregate fuel' = gabungan/total dari beberapa 'Fuel' individual.
  Contoh: 'Renewables' = 'Wind and Solar' + 'Hydro, Bioenergy and Other Renewables'
  Contoh: 'Fossil' = 'Coal' + 'Gas' + 'Other Fossil'
- Jika kita jumlahkan Aggregate dan Individual bersamaan, nilainya dihitung 2x.
- Solusi: Tandai mana yang aggregate (gabungan) dan mana individual (satuan).

In [ ]:
# Daftar variable yang merupakan AGREGASI (gabungan dari beberapa fuel)
daftar_variabel_agregat = [
    'Clean',                                    # = Nuclear + Renewables
    'Fossil',                                   # = Coal + Gas + Other Fossil
    'Gas and Other Fossil',                     # = Gas + Other Fossil
    'Hydro, Bioenergy and Other Renewables',    # = Hydro + Bioenergy + Other Renewables
    'Renewables',                               # = Wind + Solar + Hydro + Bioenergy + Other Renewables
    'Wind and Solar'                            # = Wind + Solar
]

# Daftar variable yang merupakan INDIVIDUAL (bahan bakar satuan)
daftar_variabel_individual = [
    'Bioenergy', 'Coal', 'Gas', 'Hydro', 'Nuclear',
    'Other Fossil', 'Solar', 'Wind', 'Other Renewables'
]

# Buat kolom baru: True jika baris ini adalah data agregat
data_mentah['Adalah_Agregat'] = data_mentah['Variable'].isin(daftar_variabel_agregat)

jumlah_agregat = data_mentah['Adalah_Agregat'].sum()
jumlah_individual = (~data_mentah['Adalah_Agregat']).sum()
print(f"Baris Aggregate fuel  : {jumlah_agregat:,}")
print(f"Baris Individual/lain : {jumlah_individual:,}")

Verifikasi dengan contoh data

In [ ]:
# Ambil sample satu negara dan satu tanggal untuk membuktikan hubungan aggregate-individual
negara_contoh = 'Australia'
tanggal_contoh = '2024-01-01'
filter_contoh = (
    (data_mentah['Area'] == negara_contoh) &
    (data_mentah['Date'] == tanggal_contoh) &
    (data_mentah['Unit'] == 'TWh')
)
data_contoh = data_mentah[filter_contoh][['Variable', 'Subcategory', 'Value', 'Adalah_Agregat']]
print(f"\nContoh data {negara_contoh} ({tanggal_contoh}) dalam TWh:")
print(data_contoh.to_string())
print("\n>> Untuk analisis, gunakan 'Fuel' (individual) ATAU 'Aggregate fuel', jangan dicampur.")

### Memisahkan Country dan Region

Penjelasan:
- Kolom 'Area type' berisi 2 nilai: 'Country or economy' dan 'Region'.
- Region = kelompok negara seperti ASEAN, Asia, EU, Europe, G20, G7,
           Latin America and Caribbean, North America, Oceania, OECD, World.
- Dipisahkan agar analisis per negara tidak tercampur dengan data
  gabungan regional (yang bisa menyebabkan double counting juga).

In [ ]:
# Pisahkan data berdasarkan tipe area
data_negara = data_mentah[data_mentah['Area type'] == 'Country or economy'].copy()
data_region = data_mentah[data_mentah['Area type'] == 'Region'].copy()

print(f"Total data        : {len(data_mentah):,} baris")
print(f"Data Country      : {len(data_negara):,} baris ({data_negara['Area'].nunique()} negara)")
print(f"Data Region       : {len(data_region):,} baris ({data_region['Area'].nunique()} region)")
print(f"\nDaftar Region     : {data_region['Area'].unique().tolist()}")
print(f"Sample Country    : {data_negara['Area'].unique()[:10].tolist()} ...")

### Buat label (PROVISIONAL)

Penjelasan:
- Dataset tidak memiliki kolom 'provisional' secara eksplisit.
- Kita buat label berdasarkan heuristik: data dari 12 bulan terakhir
  (dihitung dari tanggal terbaru tiap negara) dianggap 'provisional'
  karena sering direvisi oleh Ember.
- Data yang lebih lama dari 12 bulan dianggap sudah 'confirmed/final'.

In [ ]:
# Cari tanggal terbaru untuk setiap area/negara
tanggal_terbaru_per_area = data_mentah.groupby('Area')['Date'].max().reset_index()
tanggal_terbaru_per_area.columns = ['Area', 'Tanggal_Terbaru']

# Hitung batas provisional: 12 bulan sebelum tanggal terbaru masing-masing area
tanggal_terbaru_per_area['Batas_Provisional'] = (
    tanggal_terbaru_per_area['Tanggal_Terbaru'] - pd.DateOffset(months=12)
)

# Gabungkan batas provisional ke data utama
data_mentah = data_mentah.merge(
    tanggal_terbaru_per_area[['Area', 'Batas_Provisional']],
    on='Area',
    how='left'
)

In [ ]:

# Tandai: True jika tanggal data >= batas provisional (artinya data masih baru/sementara)
data_mentah['Adalah_Provisional'] = data_mentah['Date'] >= data_mentah['Batas_Provisional']

# Hapus kolom bantu yang tidak perlu lagi
data_mentah = data_mentah.drop(columns=['Batas_Provisional'])

jumlah_provisional = data_mentah['Adalah_Provisional'].sum()
jumlah_confirmed = (~data_mentah['Adalah_Provisional']).sum()
print(f"Data Provisional  : {jumlah_provisional:,} baris ({jumlah_provisional/len(data_mentah)*100:.1f}%)")
print(f"Data Confirmed    : {jumlah_confirmed:,} baris ({jumlah_confirmed/len(data_mentah)*100:.1f}%)")

# Perbarui dataframe terpisah setelah penambahan kolom baru
data_negara = data_mentah[data_mentah['Area type'] == 'Country or economy'].copy()
data_region = data_mentah[data_mentah['Area type'] == 'Region'].copy()


### Pengecekan ketersediaan data

Penjelasan:
- Mengecek berapa bulan data tersedia per negara.
- Melihat negara dengan data terlengkap vs paling sedikit.
- Penting untuk mengetahui apakah ada negara yang datanya
  terlalu sedikit untuk analisis yang reliable.

In [ ]:
# Hitung ketersediaan data per negara
ketersediaan = data_negara.groupby('Area').agg(
    Tanggal_Awal=('Date', 'min'),        # Tanggal data pertama
    Tanggal_Akhir=('Date', 'max'),       # Tanggal data terakhir
    Jumlah_Bulan=('YearMonth', 'nunique'),  # Berapa bulan unik yang tersedia
    Jumlah_Baris=('Date', 'count')       # Total baris data
).sort_values('Jumlah_Bulan', ascending=False)

print("Top 10 negara dengan data terbanyak (bulan):")
print(ketersediaan.head(10).to_string())

print("\nBottom 10 negara dengan data paling sedikit (bulan):")
print(ketersediaan.tail(10).to_string())

In [ ]:
# Hitung rata-rata dan median ketersediaan
rata_rata_bulan = ketersediaan['Jumlah_Bulan'].mean()
median_bulan = ketersediaan['Jumlah_Bulan'].median()
print(f"\nRata-rata ketersediaan: {rata_rata_bulan:.0f} bulan per negara")
print(f"Median ketersediaan  : {median_bulan:.0f} bulan per negara")

# Cek berapa banyak nilai 'Value' yang kosong di data negara
jumlah_value_kosong = data_negara['Value'].isnull().sum()
total_baris_negara = len(data_negara)
print(f"\nMissing 'Value' di data Country: {jumlah_value_kosong:,} dari {total_baris_negara:,} ({jumlah_value_kosong/total_baris_negara*100:.2f}%)")

### Cek outlier

Penjelasan:
- Metode IQR (Interquartile Range) digunakan untuk mendeteksi outlier.

  Cara kerja IQR:
  1. Q1 (Kuartil 1) = nilai di persentil ke-25 (25% data di bawahnya)
  2. Q3 (Kuartil 3) = nilai di persentil ke-75 (75% data di bawahnya)
  3. IQR = Q3 - Q1  (rentang antar kuartil, mencakup 50% data tengah)
  4. Batas bawah = Q1 - 1.5 * IQR
  5. Batas atas  = Q3 + 1.5 * IQR
  6. Data di luar batas bawah atau batas atas dianggap OUTLIER.

- Pengecekan dilakukan per kelompok Variable + Unit agar adil
  (misal: TWh dan % punya skala berbeda).
- Outlier TIDAK dihapus karena bisa jadi data valid dari negara besar
  (China, US, India yang memang produksi listriknya sangat besar).

In [ ]:
# Ambil hanya data negara yang kolom Value-nya tidak kosong
data_untuk_cek_outlier = data_negara[data_negara['Value'].notna()].copy()

# List untuk menyimpan hasil deteksi outlier per kelompok
hasil_deteksi_outlier = []

# Loop per kelompok Variable + Unit
for (nama_variabel, satuan), kelompok in data_untuk_cek_outlier.groupby(['Variable', 'Unit']):

    # Q1 = Kuartil 1 (persentil ke-25), batas bawah dari 50% data tengah
    kuartil_1 = kelompok['Value'].quantile(0.25)

    # Q3 = Kuartil 3 (persentil ke-75), batas atas dari 50% data tengah
    kuartil_3 = kelompok['Value'].quantile(0.75)

    # IQR = Rentang Antar Kuartil, mengukur sebaran 50% data tengah
    rentang_antar_kuartil = kuartil_3 - kuartil_1

    # Hitung batas bawah dan atas untuk deteksi outlier
    batas_bawah = kuartil_1 - 1.5 * rentang_antar_kuartil
    batas_atas = kuartil_3 + 1.5 * rentang_antar_kuartil

    # Hitung jumlah data yang berada di luar batas (= outlier)
    jumlah_outlier = (
        (kelompok['Value'] < batas_bawah) | (kelompok['Value'] > batas_atas)
    ).sum()

In [ ]:

    # Simpan hasil jika ada outlier
    if jumlah_outlier > 0:
        hasil_deteksi_outlier.append({
            'Variable': nama_variabel,
            'Unit': satuan,
            'Jumlah_Data': len(kelompok),
            'Jumlah_Outlier': jumlah_outlier,
            'Persen_Outlier': round(jumlah_outlier / len(kelompok) * 100, 2),
            'Batas_Bawah': round(batas_bawah, 2),
            'Batas_Atas': round(batas_atas, 2)
        })

# Buat DataFrame dari hasil deteksi, urutkan dari outlier terbanyak
tabel_outlier = pd.DataFrame(hasil_deteksi_outlier).sort_values('Jumlah_Outlier', ascending=False)

print("Ringkasan outlier per Variable-Unit (metode IQR):")
print(tabel_outlier.to_string(index=False))

total_outlier = tabel_outlier['Jumlah_Outlier'].sum()
print(f"\nTotal data outlier : {total_outlier:,}")
print(">> Outlier tidak dihapus karena bisa jadi valid (negara besar seperti China, US, India).")

### Ringkasan akhir dan simpan data baru

Penjelasan:
- Menyimpan 3 file output:
  1. df_clean.csv         = data gabungan (country + region) yang sudah diproses
  2. df_country_clean.csv = hanya data negara
  3. df_region_clean.csv  = hanya data region
- Ringkasan statistik akhir ditampilkan.

In [ ]:
# Tampilkan ringkasan akhir
print("=" * 40)
print("  RINGKASAN PREPROCESSING")
print("=" * 40)
print(f"  Baris awal          : 501,483")
print(f"  Baris akhir         : {len(data_mentah):,}")
print(f"  Duplikat dihapus    : {jumlah_duplikat}")
print(f"  Kolom awal          : 18")
print(f"  Kolom akhir         : {len(data_mentah.columns)} (+Year, Month, Quarter, YearMonth, Adalah_Agregat, Adalah_Provisional)")
print(f"  Jumlah negara       : {data_negara['Area'].nunique()}")
print(f"  Jumlah region       : {data_region['Area'].nunique()}")
print(f"  Rentang waktu       : {data_mentah['Date'].min().date()} s/d {data_mentah['Date'].max().date()}")
print(f"  Data provisional    : {jumlah_provisional:,} baris")
print(f"  Data confirmed      : {jumlah_confirmed:,} baris")
print("=" * 40)

# Simpan hasil preprocessing ke file CSV
data_mentah.to_csv('dataset/df_clean.csv', index=False)
data_negara.to_csv('dataset/df_country_clean.csv', index=False)
data_region.to_csv('dataset/df_region_clean.csv', index=False)

print(f"\nFile tersimpan:")
print(f"  - dataset/df_clean.csv          ({len(data_mentah):,} baris)")
print(f"  - dataset/df_country_clean.csv  ({len(data_negara):,} baris)")
print(f"  - dataset/df_region_clean.csv   ({len(data_region):,} baris)")

print("\n" + "=" * 60)
print("PREPROCESSING SELESAI!")
print("=" * 60)

In [ ]:
print("=" * 20)
print("    ASTAGA REG")
print("    MANTAP QUEEN")
print("=" * 20)